<a href="https://colab.research.google.com/github/nijatbayramoov/Movies/blob/main/Nijat_Bayramov_Movies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring and Analyzing Movie Ratings with Polars: Project



| File        | Column     | Description                                                                 |
|------------|-----------|------------------------------------------------------------------------------|
| movies.csv | movieId   | ID number that uniquely identifies each movie                                |
| movies.csv | title     | Name of the movie (may include year in parentheses)                          |
| movies.csv | genres    | Pipe-separated list of genres (e.g., Comedy|Drama)                           |
| ratings.csv| userId    | ID number that uniquely identifies each user                                 |
| ratings.csv| movieId   | ID number that uniquely identifies each movie                                |
| ratings.csv| rating    | Rating score from 0.5 to 5.0 (half-star increments allowed)                  |
| ratings.csv| timestamp | Unix timestamp when the rating was given                                     |
| tags.csv   | userId    | ID number that uniquely identifies each user                                 |
| tags.csv   | movieId   | ID number that uniquely identifies each movie                                |
| tags.csv   | tag       | User-generated tag (string, e.g., 'funny')                                    |
| tags.csv   | timestamp | Unix timestamp when the tag was added                                        |


## Step 1: Load and inspect data

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
movies = pl.read_csv('/content/drive/MyDrive/Dataset/Movies polaris/movies.csv')
ratings = pl.read_csv('/content/drive/MyDrive/Dataset/Movies polaris/ratings.csv')
tags = pl.read_csv('/content/drive/MyDrive/Dataset/Movies polaris/tags.csv')

In [ ]:
print(movies.head())
print(ratings.head())
print(tags.head())

shape: (5, 3)
┌─────────┬─────────────────────────────────┬─────────────────────────────────┐
│ movieId ┆ title                           ┆ genres                          │
│ ---     ┆ ---                             ┆ ---                             │
│ i64     ┆ str                             ┆ str                             │
╞═════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 1       ┆ Toy Story (1995)                ┆ Adventure|Animation|Children|C… │
│ 2       ┆ Jumanji (1995)                  ┆ Adventure|Children|Fantasy      │
│ 3       ┆ Grumpier Old Men (1995)         ┆ Comedy|Romance                  │
│ 4       ┆ Waiting to Exhale (1995)        ┆ Comedy|Drama|Romance            │
│ 5       ┆ Father of the Bride Part II (1… ┆ Comedy                          │
└─────────┴─────────────────────────────────┴─────────────────────────────────┘
shape: (5, 4)
┌────────┬─────────┬────────┬───────────┐
│ userId ┆ movieId ┆ rating ┆ timestamp │
│ ---   

In [ ]:
print("movies shape:", movies.shape)
print("ratings shape:", ratings.shape)
print("tags shape:", tags.shape)

movies shape: (9742, 3)
ratings shape: (100836, 4)
tags shape: (3683, 4)


## Step 2: Preprocess data

### Data tiplərinin yoxlanması

`movies` cədvəlindəki `movieId` sütunu tam ədəd (`i64`) olmalıdır, `genres` sütunu isə mətn (`str`). `ratings` cədvəlindəki `rating` sütunu onluq ədəd (`f64`), `timestamp` isə tam ədəd (`i64`) olmalıdır. Aşağıda bütün sütunların tipləri yoxlanır.

In [ ]:
print("movies dtypes:", movies.dtypes)
print("ratings dtypes:", ratings.dtypes)
print("tags dtypes:", tags.dtypes)

movies dtypes: [Int64, String, String]
ratings dtypes: [Int64, Int64, Float64, Int64]
tags dtypes: [Int64, Int64, String, Int64]


Bütün sütunların tipləri düzgündür - əlavə konversiya tələb olunmur.

### Boş dəyərlərin yoxlanması

Boş dəyərlər (`null`) analiz nəticələrini təhrif edə bilər, ona görə hər cədvəldə boş dəyərləri yoxlayırıq.

In [ ]:
print(movies.null_count())
print(ratings.null_count())
print(tags.null_count())

shape: (1, 3)
┌─────────┬───────┬────────┐
│ movieId ┆ title ┆ genres │
│ ---     ┆ ---   ┆ ---    │
│ u32     ┆ u32   ┆ u32    │
╞═════════╪═══════╪════════╡
│ 0       ┆ 0     ┆ 0      │
└─────────┴───────┴────────┘
shape: (1, 4)
┌────────┬─────────┬────────┬───────────┐
│ userId ┆ movieId ┆ rating ┆ timestamp │
│ ---    ┆ ---     ┆ ---    ┆ ---       │
│ u32    ┆ u32     ┆ u32    ┆ u32       │
╞════════╪═════════╪════════╪═══════════╡
│ 0      ┆ 0       ┆ 0      ┆ 0         │
└────────┴─────────┴────────┴───────────┘
shape: (1, 4)
┌────────┬─────────┬─────┬───────────┐
│ userId ┆ movieId ┆ tag ┆ timestamp │
│ ---    ┆ ---     ┆ --- ┆ ---       │
│ u32    ┆ u32     ┆ u32 ┆ u32       │
╞════════╪═════════╪═════╪═══════════╡
│ 0      ┆ 0       ┆ 0   ┆ 0         │
└────────┴─────────┴─────┴───────────┘


Heç bir cədvəldə boş dəyər yoxdur — dataseti bu baxımdan təmizdir.

### Dublikat dəyərlərin yoxlanması

Dublikat sətirlər ortalama hesablamalarda eyni məlumatı iki dəfə saymağa səbəb ola bilər. Hər cədvəl üçün tam dublikatları yoxlayırıq.

In [ ]:
print("movies duplicates:", movies.is_duplicated().sum())
print("ratings duplicates:", ratings.is_duplicated().sum())
print("tags duplicates:", tags.is_duplicated().sum())

movies duplicates: 0
ratings duplicates: 0
tags duplicates: 0


Heç bir cədvəldə tam dublikat sətir tapılmadı. Dataseti analiz üçün hazırdır.

**Ümumi qiymətləndirmə:** Bu dataset əvvəlcədən təmizlənmiş görünür — boş dəyər, tip uyuşmazlığı və dublikat yoxdur. Real dünya datasetlərində isə adətən bu problemlər mövcud olur (məsələn, istifadəçi iki dəfə eyni filmi reytinq vermişsə, dublikat yarana bilər).

## [A] Easy

### A1: Verify that values in the 'rating' column in the ratings table are sensible (i.e., ranges from 0.5 to 5.0). Print the min and max values.

In [ ]:
rating_range = ratings.select([
    pl.col('rating').min().alias('min_rating'),
    pl.col('rating').max().alias('max_rating')
])
rating_range

min_rating,max_rating
f64,f64
0.5,5.0


This analysis checks the minimum and maximum values in the rating column to verify that the ratings are within the expected range.

### A2: Compute and print the count of ratings for each score (0.5 through 5.0) in a table format.

In [ ]:
rating_counts=ratings.group_by('rating').count().sort('rating',descending=True).head(2)
rating_counts

/tmp/ipykernel_5867/542733641.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  rating_counts=ratings.group_by('rating').count().sort('rating',descending=True).head(2)


rating,count
f64,u32
5.0,13211
4.5,8551


This analysis counts how many times each rating value appears and displays the most frequent rating scores.

### A3: Compute and print the average rating per genre (explode the genres_list column and group by genre).

In [ ]:
movies_with_genres = movies.with_columns(
    pl.col('genres').str.split('|').alias('genres_list')
)

avg_rating_per_genre = (
    movies_with_genres
    .explode('genres_list')
    .join(ratings, on='movieId')
    .group_by('genres_list')
    .agg(pl.col('rating').mean().alias('avg_rating'))
    .sort('avg_rating', descending=True)
)
avg_rating_per_genre

genres_list,avg_rating
str,f64
"""Film-Noir""",3.920115
"""War""",3.808294
"""Documentary""",3.797785
"""Crime""",3.658294
"""Drama""",3.656184
…,…
"""Sci-Fi""",3.455721
"""Action""",3.447984
"""Children""",3.412956


This analysis calculates the average rating for each movie genre by splitting and expanding the genres column into separate genre entries.

### A4: Compute and print the min, max, and mean number of ratings per user.

In [ ]:
ratings_per_user = ratings.group_by('userId').agg(pl.len().alias('num_ratings'))

user_stats = ratings_per_user.select([
    pl.col('num_ratings').min().alias('min_ratings'),
    pl.col('num_ratings').max().alias('max_ratings'),
    pl.col('num_ratings').mean().alias('mean_ratings')
])
user_stats

min_ratings,max_ratings,mean_ratings
u32,u32,f64
20,2698,165.304918


This analysis computes the total number of ratings, minimum rating, maximum rating, and average rating for each user.

### A5: Find and print the top 5 users with the most ratings.

In [ ]:
top5_users = (
    ratings
    .group_by('userId')
    .agg(pl.len().alias('rating_count'))
    .sort('rating_count', descending=True)
    .head(5)
)
top5_users

userId,rating_count
i64,u32
414,2698
599,2478
474,2108
448,1864
274,1346


This analysis identifies the top 5 users who have given the highest number of ratings.

## [B] Medium

### B1: Compute and print the average rating of the top 10 most frequently rated movies vs. the bottom 10 (movies with the fewest ratings, but at least 5 ratings).

In [ ]:
movie_rating_stats = (
    ratings
    .group_by('movieId')
    .agg([
        pl.len().alias('rating_count'),
        pl.col('rating').mean().alias('avg_rating')
    ])
)
top10 = movie_rating_stats.sort('rating_count', descending=True).head(10)
bottom10 = (
    movie_rating_stats
    .filter(pl.col('rating_count') >= 5)
    .sort('rating_count')
    .head(10)
)

print("Top 10 most rated movies - average rating:", round(top10['avg_rating'].mean(), 4))
print("Bottom 10 least rated movies (min 5 ratings) - average rating:", round(bottom10['avg_rating'].mean(), 4))

Top 10 most rated movies - average rating: 4.1353
Bottom 10 least rated movies (min 5 ratings) - average rating: 3.19


This code groups movies by their IDs, calculates the number of ratings and average rating for each movie, then compares the average ratings of the top 10 most rated movies and the bottom 10 least rated movies.

### B2: Compute and print a table showing the distribution of the number of ratings per movie.

In [ ]:
ratings_distribution = (
    movie_rating_stats
    .group_by('rating_count')
    .agg(pl.len().alias('num_movies'))
    .sort('rating_count')
)
ratings_distribution

rating_count,num_movies
u32,u32
1,3446
2,1298
3,800
4,530
5,382
…,…
278,1
279,1
307,1


This code analyzes the distribution of movie ratings by counting how many movies have the same number of ratings and sorting the results by rating count.

### B3: Print the top 20 most frequently rated movies (display their id, title, and rating count).

In [ ]:
top20_movies = (
    movie_rating_stats
    .join(movies.select(['movieId', 'title']), on='movieId')
    .sort('rating_count', descending=True)
    .head(20)
    .select(['movieId', 'title', 'rating_count'])
)
top20_movies

movieId,title,rating_count
i64,str,u32
356,"""Forrest Gump (1994)""",329
318,"""Shawshank Redemption, The (199…",317
296,"""Pulp Fiction (1994)""",307
593,"""Silence of the Lambs, The (199…",279
2571,"""Matrix, The (1999)""",278
…,…,…
47,"""Seven (a.k.a. Se7en) (1995)""",203
780,"""Independence Day (a.k.a. ID4) …",202
150,"""Apollo 13 (1995)""",201


This analysis finds the top 20 most frequently rated movies by counting the number of ratings each movie received.

### B4: Compute and print the total number of unique genres and list the top 10 most common genres with their counts.

In [ ]:
genres_exploded = movies_with_genres.explode('genres_list')

unique_genre_count = genres_exploded['genres_list'].n_unique()
print("Total unique genres:", unique_genre_count)

top10_genres = (
    genres_exploded
    .group_by('genres_list')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .head(10)
)

top10_genres

Total unique genres: 20


genres_list,count
str,u32
"""Drama""",4361
"""Comedy""",3756
"""Thriller""",1894
"""Action""",1828
"""Romance""",1596
"""Adventure""",1263
"""Crime""",1199
"""Sci-Fi""",980
"""Horror""",978


This analysis shows which movie genres receive the most ratings by counting ratings for each genre and sorting them in descending order.

### B5: For each genre, compute and print the number of movies and average rating in a sorted table.

In [ ]:
genre_stats = (
    genres_exploded
    .join(ratings, on='movieId')
    .group_by('genres_list')
    .agg([
        pl.col('movieId').n_unique().alias('num_movies'),
        pl.col('rating').mean().alias('avg_rating')
    ])
    .sort('num_movies', descending=True)
)
genre_stats

genres_list,num_movies,avg_rating
str,u32,f64
"""Drama""",4349,3.656184
"""Comedy""",3753,3.384721
"""Thriller""",1889,3.493706
"""Action""",1828,3.447984
"""Romance""",1591,3.506511
…,…,…
"""Musical""",333,3.563678
"""Western""",167,3.583938
"""IMAX""",158,3.618335


This analysis helps us understand which movie genres are the most popular and how highly they are rated by users. The count shows how many ratings each genre received, while the mean shows the average user rating for that genre.

## [C] Hard

### C1: Compute and print the average number of tags per movie, and show the min and max tags per movie.

In [ ]:
tags_per_movie = tags.group_by('movieId').agg(pl.len().alias('tag_count'))

tag_stats = tags_per_movie.select([
    pl.col('tag_count').mean().alias('avg_tags'),
    pl.col('tag_count').min().alias('min_tags'),
    pl.col('tag_count').max().alias('max_tags')
])
tag_stats

avg_tags,min_tags,max_tags
f64,u32,u32
2.342875,1,181


This analysis shows how many tags each movie has and computes the minimum, maximum, and average number of tags per movie.

### C2: Print the top 20 tags that are applied most frequently (display the tag and count).

In [ ]:
top20_tags = (
    tags
    .group_by('tag')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .head(20)
)
top20_tags

tag,count
str,u32
"""In Netflix queue""",131
"""atmospheric""",36
"""thought-provoking""",24
"""superhero""",24
"""surreal""",23
…,…
"""visually appealing""",19
"""politics""",18
"""mental illness""",16


This analysis counts how many times each tag was used and displays the most frequently applied tags.

### C3: For each movie, compute the proportion of its ratings that are 4 or higher, and print a table with movie ID, title, and high_rating_proportion (for movies with at least 5 ratings).

In [ ]:
high_rating_prop = (
    ratings
    .group_by('movieId')
    .agg([
        pl.len().alias('total_ratings'),
        (pl.col('rating') >= 4).sum().alias('high_ratings')
    ])
    .filter(pl.col('total_ratings') >= 5)
    .with_columns(
        (pl.col('high_ratings') / pl.col('total_ratings')).alias('high_rating_proportion')
    )
    .join(movies.select(['movieId', 'title']), on='movieId')
    .select(['movieId', 'title', 'high_rating_proportion'])
    .sort('high_rating_proportion', descending=True)
)
high_rating_prop

movieId,title,high_rating_proportion
i64,str,f64
942,"""Laura (1944)""",1.0
1041,"""Secrets & Lies (1996)""",1.0
1192,"""Paris Is Burning (1990)""",1.0
1446,"""Kolya (Kolja) (1996)""",1.0
1931,"""Mutiny on the Bounty (1935)""",1.0
…,…,…
108945,"""RoboCop (2014)""",0.0
112788,"""Sex Tape (2014)""",0.0
117895,"""Maze Runner: Scorch Trials (20…",0.0


### C4: For each user, compute the proportion of their ratings that are re-ratings (multiple ratings to the same movie), and print the top 10 users with the highest re-rating proportion.

In [ ]:
re_ratings = (
    ratings
    .group_by(['userId', 'movieId'])
    .agg(pl.len().alias('ratings_per_movie'))
)

re_rating_prop = (
    re_ratings
    .group_by('userId')
    .agg([
        pl.len().alias('total_movies'),
        (pl.col('ratings_per_movie') > 1).sum().alias('re_rated_movies')
    ])
    .with_columns(
        (pl.col('re_rated_movies') / pl.col('total_movies')).alias('re_rating_proportion')
    )
    .sort('re_rating_proportion', descending=True)
    .head(10)
)
re_rating_prop

userId,total_movies,re_rated_movies,re_rating_proportion
i64,u32,u32,f64
498,35,0,0.0
257,20,0,0.0
358,41,0,0.0
183,57,0,0.0
555,578,0,0.0
304,216,0,0.0
248,51,0,0.0
268,129,0,0.0
149,58,0,0.0


### C5: Print the top 20 movies with the highest average ratings (display movie IDs, titles, and average ratings, with at least 10 ratings per movie).

In [ ]:
top20_avg_rating = (
    ratings
    .group_by('movieId')
    .agg([
        pl.len().alias('rating_count'),
        pl.col('rating').mean().alias('avg_rating')
    ])
    .filter(pl.col('rating_count') >= 10)
    .sort('avg_rating', descending=True)
    .head(20)
    .join(movies.select(['movieId', 'title']), on='movieId')
    .select(['movieId', 'title', 'avg_rating'])
)
top20_avg_rating

movieId,title,avg_rating
i64,str,f64
176,"""Living in Oblivion (1995)""",4.307692
318,"""Shawshank Redemption, The (199…",4.429022
898,"""Philadelphia Story, The (1940)""",4.310345
905,"""It Happened One Night (1934)""",4.321429
922,"""Sunset Blvd. (a.k.a. Sunset Bo…",4.333333
…,…,…
3451,"""Guess Who's Coming to Dinner (…",4.545455
3468,"""Hustler, The (1961)""",4.333333
7156,"""Fog of War: Eleven Lessons fro…",4.307692


This analysis finds the top 20 movies with the highest average ratings among movies that received at least 10 ratings.

### C6: Compute and print the correlation between the number of ratings a movie receives and its average rating.

In [ ]:
correlation = movie_rating_stats.select(
    pl.corr('rating_count', 'avg_rating').alias('correlation')
)

correlation

correlation
f64
0.127259


This analysis measures the correlation between the number of ratings a movie receives and its average rating to understand whether more popular movies tend to have higher ratings.